In [2]:
import pandas as pd 
from transformers import T5Tokenizer , Trainer , TrainingArguments , T5ForConditionalGeneration

/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

In [4]:
train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [5]:
train_data.shape

(14732, 3)

In [6]:
val_data.shape

(818, 3)

In [7]:
#random sampling 

train_data = train_data.sample(n=4000 , random_state=42).reset_index(drop=True)
vsl_data = val_data.sample(n=500 , random_state=42).reset_index(drop=True)

In [8]:
train_data.shape

(4000, 3)

Data pre-processing 

In [9]:
import re 

def clean_data(text):
    text= re.sub(r"\r\n" , " " , text) #line
    text= re.sub(r"\s+n" , " " , text) #spaces
    text= re.sub(r"<.*?>" , " " , text) #html tag 
    text = text.strip().lower()
    return text

In [10]:
train_data['dialogue']= train_data['dialogue'].apply(clean_data)
train_data['summary']= train_data['summary'].apply(clean_data)

val_data['dialogue']= val_data['dialogue'].apply(clean_data)
val_data['summary']= val_data['summary'].apply(clean_data)

In [11]:
train_data['dialogue'][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

Tokenize

In [12]:
tokanizer = T5Tokenizer.from_pretrained("t5-small")

In [29]:
#row data -- tokanizd inputs for fine-tunig 

def tokanize(data):
    inputs= tokanizer(data['dialogue'] , padding="max_length" , max_length=512 , truncation=True)
    targets= tokanizer(data['summary'] , padding="max_length" , max_length=150 , truncation=True)

    inputs["labels"] = targets["input_ids"] # tokan ids => add to input as label
    return inputs


In [30]:
train_dataset=train_data.apply(tokanize , axis=1).tolist()
val_dataset = val_data.apply(tokanize , axis=1).tolist()

In [31]:
train_dataset

[{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [32]:
# input ids - dialogue => token ids 

# 1=> End of  , 0=> padding

# attention mask 

# labels - taget => summary tokan

#working with our model 


In [33]:
#NLP => generation task 

model = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights: 100%|██████████| 131/131 [00:00<00:00, 5140.46it/s]


In [34]:
import torch 

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.s_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

print("device : " , device)
model.to(device)           

device :  mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [35]:
#training aruguments 

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size =8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500 
    
)

In [36]:
trainer  = Trainer(
    model = model , 
    args = training_args,
    train_dataset= train_dataset,
    eval_dataset = val_dataset
)

In [37]:
#train the model 
trainer.train()

Epoch,Training Loss,Validation Loss
1,3.586309,0.388052
2,0.410577,0.362376
3,0.385674,0.355588
4,0.373450,0.351811
5,0.365738,0.350390
6,0.362041,0.350136


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.04it/s]
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.00it/s]
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.12it/s]
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.38it/s]
/opt/anaconda3/envs/myenv/

TrainOutput(global_step=3000, training_loss=0.9139648742675781, metrics={'train_runtime': 2633.3726, 'train_samples_per_second': 9.114, 'train_steps_per_second': 1.139, 'total_flos': 3248203235328000.0, 'train_loss': 0.9139648742675781, 'epoch': 6.0})

In [ ]:
#model => finr-tune=> save the load 

In [40]:
model.save_pretrained("./saved_summary_model")
tokanizer.save_pretrained("./saved_summary_model")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.37s/it]


('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

test the core logic fro summarization 

In [43]:
model= T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokanizer= T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 4567.33it/s]


In [61]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue)

    #tokanize
    inputs = tokanizer(
        dialogue,
        padding = "max_length",
        max_length = 512,
        truncation = True,
        return_tensors = "pt"
    ).to(device)

    # generate the summary => tokan ids 
    model.to(device)
    targets = model.generate(
        input_ids = inputs["input_ids"],
        attention_mask= inputs["attention_mask"],
        max_length=150,
        num_beams= 4, #compare 4 summary
        early_stopping = True
    )

    #tokan ids convert to summary => decode

    summary = tokanizer.decode(targets[0], skip_special_tokens=True)
    return summary

In [ ]:
test_dialouge = """
Emma: Hey, did you hear back from the landlord about the apartment?
Jake: Yeah, he said the lease is ready whenever we want to sign
Emma: That's great! Did he mention anything about the deposit?
Jake: One month's rent, same as he told us before
Emma: Ok good, no surprises there. When can we actually move in?
Jake: He said the 1st of next month, but we could probably get the keys a few days earlier to start cleaning
Emma: Perfect, that gives us time before work starts again
Jake: Also, I checked with the moving company, they can do Saturday morning
Emma: Nice, how much are they charging?
Jake: Around 150 for the whole move since it's not too much stuff
Emma: That's reasonable, let's book them
Jake: Will do. Oh and I almost forgot, the internet guy is coming Tuesday to set up wifi
Emma: Good thing you remembered, I would've forgotten completely
Jake: Ha, no worries. So just to recap, we get keys a few days early, move Saturday, internet Tuesday
Emma: Sounds like a solid plan. I'll start packing this weekend
Jake: Same here, let's talk tomorrow about splitting up the packing list
Emma: Sounds good, talk tomorrow!
"""

summary = summarize_dialogue(test_dialouge)

print("summary : " , summary)

summary :  jake and emma are going to move in on ext month. they will get keys a few days early, move saturday, internet tuesday. they will split up the packing list this weekend.


: 